In [ ]:
# import the library
from pynq import Overlay     # import the overlay
from pynq import allocate    # import for CMA (contingeous memory allocation)
from pynq import DefaultIP   # import the ip connector library for extension
from pynq import Interrupt
import asyncio
import numpy as np
import os
import dfx4ml.cap         as cap
import dfx4ml.mem_alloc   as dataAlloc  # import the memory allocation library
import dfx4ml.dfx_unified as dfx_unified
import dfx4ml.dfx_mgs_debug as dfx_mgs_debug
import time

In [ ]:
PRJ_DIR = os.getcwd()
PRJ_HW_DIR = PRJ_DIR + '/hw/'
PRJ_TC_DIR = PRJ_DIR + '/sw/'
DFX_CONFIG_FILE = 'dfx_ctrl_meta.txt'

FULL_BS_NAME = 'system.bin'
PAR_BS_NAME_0 = 'par1.bin'  ###### dma to magic stream 1
PAR_BS_NAME_1 = 'par2.bin'  ###### magic stream 1 to magic stream 2
INPUT_DATA_NAME = "input_x.npy"

AMT_QUERY = 100
INPUT_SHAPE = (AMT_QUERY, 1)  # 4*4* float32 = 64 bytes
OUTPUT_SHAPE = (AMT_QUERY, 1)  # 4*4* float32 = 64 bytes
AMT_SLOT = 2

input_x = (np.arange(AMT_QUERY, dtype=np.int32)+48).reshape(INPUT_SHAPE)
np.save(os.path.join(PRJ_TC_DIR, INPUT_DATA_NAME), input_x)


In [ ]:
cap.change_pl_config_mode("pcap", True, "")

In [ ]:
#### load the overlay
overlay  = Overlay(PRJ_HW_DIR + FULL_BS_NAME)

In [ ]:
#### create the interrupt pin
overlay.interrupt_pins


In [ ]:
my_interrupt = Interrupt('dfx_unified_0/dfx_intr')  # index 0 from your mapping

In [ ]:
dfx_unifed_ip = overlay.dfx_unified_0

In [ ]:
#### get the device
dmaIp      = dfx_unifed_ip.dfx_dma
dfx_ctrl  = dfx_unifed_ip.dfx_ctrl
dfx_mng = dfx_unifed_ip.dfx_mng

In [ ]:
#### configure the dfx controller ip to match the address space
dfx_ctrl.config(PRJ_HW_DIR + DFX_CONFIG_FILE)
print("regIdxSize = ", dfx_ctrl.BLS_REGID)

In [ ]:
### change reconfigure mode
cap.change_pl_config_mode("icap", True, "")

In [ ]:
dfx_ctrl.print_status()

In [ ]:
#### shutdown all system
dfx_mng .shutdown_engine()
dfx_ctrl.shutdown_engine()

In [ ]:
# get physical address of dma and dfx controller
dmaPhyAddr     =  0x0002_0000 #overlay.ip_dict['dataMovement/axi_dma_0']['phys_addr']
dfxCtrlPhyAddr =  0 #overlay.ip_dict['PRcontroller/dfx_controller_0']['phys_addr']

print("dma physical address: ", hex(dmaPhyAddr))
print("dfx  Ctrl physical address: ", hex(dfxCtrlPhyAddr))

In [ ]:
##### initialize magic seq
print("------ before init magic seq------")
print(dfx_mng.print_debug())

print("------ init magic sequence METADATA bank 0 -------------------------")
dfx_mng.set_end_cnt(0) ### use the last index
dfx_mng.set_dma_addr(dmaPhyAddr)
dfx_mng.set_dfx_addr(dfxCtrlPhyAddr)
dfx_mng.set_intr_ena(1)
dfx_mng.set_intr(1)  # woc  command 1 to set the interrupt to 0
dfx_mng.set_round_trip(0)  # set round trip to 0, no need to wait for the dma to finish
inputX = np.load(PRJ_TC_DIR + INPUT_DATA_NAME)
if(inputX.shape != INPUT_SHAPE):
    raise Exception(f"inputX shape is {inputX} expect {INPUT_SHAPE}")

#inputX = np.random.rand(*INPUT_SHAPE).astype(np.float32)
print("-------------init all data buffer -------------")
buf_input   , buf_input_phya   , buf_input_sz    = dataAlloc.alloc_data_uint(alloc_shape= INPUT_SHAPE , alloc_type=np.int32, input_x = inputX)
buf_out     , buf_out_phy      , buf_out_sz      = dataAlloc.alloc_data_uint(alloc_shape= OUTPUT_SHAPE, alloc_type=np.int32)
buf_input.flush()
print("------------- init dfx mng ------------------")
#                         [srcPhyAddr    ,        srcSz,  dstPhyAddr,      dstSz,st,pr,loadMask, storeMask, intrMask]
dfx_mng.set_whole_slot(0, [buf_input_phya, buf_input_sz,           0,          0, 0, 0,  0b0001, 0b0010, 0])

print("------------- after init magic seq------")
print(dfx_mng.print_debug())
d0_ip_buf, d0_addr, d0_size = \
    dfx_ctrl.allocate_bit_stream_cma(PRJ_HW_DIR + PAR_BS_NAME_0)
dfx_ctrl.set_simple_meta_data(0, d0_addr, d0_size)
dfx_ctrl.trig(0)
dfx_ctrl.restart_no_status()
dfx_ctrl.print_status()
##### start dfx controller3
async def startExecAndWait4Intr():
    start_time = time.perf_counter()  # Start timing
    dfx_mng.clear_intr()
    dfx_mng.start_engine()
    while True:
        await my_interrupt.wait()
        end_time = time.perf_counter()
        print("interrupt")
        print(f"Elapsed time: {end_time - start_time:.6f} seconds")
        break
loop2 = asyncio.get_event_loop()
task2 = loop2.create_task(startExecAndWait4Intr())
loop2.run_until_complete(task2)
print(dfx_mng.print_debug())
dfx_mng.shutdown_engine()


In [ ]:
buf_out     , buf_out_phy      , buf_out_sz      = dataAlloc.alloc_data_uint(alloc_shape= OUTPUT_SHAPE, alloc_type=np.int32)
dfx_mng.set_whole_slot(0, [             0,            0, buf_out_phy, buf_out_sz, 0, 0,  0b0010, 0b0001, 0])
print(dfx_mng.print_debug())
######## set trigger 1
d1_ip_buf, d1_addr, d1_size = \
    dfx_ctrl.allocate_bit_stream_cma(PRJ_HW_DIR + PAR_BS_NAME_1)
dfx_ctrl.set_simple_meta_data(0, d1_addr, d1_size)
dfx_ctrl.print_simple_meta_data(0)
dfx_ctrl.restart_no_status()
dfx_ctrl.print_status()
##### start dfx controller3
async def startExecAndWait4Intr2():
    start_time = time.perf_counter()  # Start timing
    dfx_mng.clear_intr()
    dfx_mng.start_engine()
    while True:
        await my_interrupt.wait()
        end_time = time.perf_counter()
        print("interrupt")
        print(f"Elapsed time: {end_time - start_time:.6f} seconds")
        break
loop2 = asyncio.get_event_loop()
task2 = loop2.create_task(startExecAndWait4Intr2())
loop2.run_until_complete(task2)
print(dfx_mng.print_debug())
dfx_mng.shutdown_engine()
print(dfx_mng.print_debug())


In [ ]:
buf_out.invalidate()
np_parRes = np.array(buf_out, dtype=np.int32)
print(np_parRes)

In [ ]:
print(f"Compare input_x and np_parRes: {np.array_equal(input_x, np_parRes)}")

In [ ]:
mgs_dbg = dfx_mgs_debug.DFX_mgs_debug(overlay.axi_gpio_0)
mgs_dbg.read_mgs_meta()